# GraphRAG System với dữ liệu JSON Sách Giáo Khoa

Notebook này sử dụng file `SGK_Lich_Su_12_Ket_Noi_Tri_Thuc.json` làm knowledge base thay vì file txt.

## Cấu trúc dữ liệu mới:
- `topic_id`: Chủ đề (vd: "Chủ đề 1")
- `topic_description`: Mô tả chủ đề
- `lesson_id`: Bài học (vd: "Bài 1")
- `lesson_title`: Tiêu đề bài học
- `sections`: Các phần chính
  - `index`: Số thứ tự
  - `title`: Tiêu đề phần
  - `subsections`: Các tiểu mục
    - `label`: Nhãn (a, b, c...)
    - `title`: Tiêu đề tiểu mục
    - `content`: Danh sách các đoạn văn

In [1]:
# Kiểm tra card GPU và CUDA
!nvidia-smi
!nvcc --version || true

Fri Dec 12 10:03:26 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cài đặt các thư viện cần thiết
import os, sys
try:
    import langchain
except Exception:
    !pip install -q langchain transformers sentence-transformers faiss-cpu accelerate bitsandbytes
    !pip install -q "huggingface_hub>=0.14.1"

In [3]:
!pip install faiss-cpu langchain-community "sentence-transformers" "faiss-cpu" transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.3/475.3 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207

In [4]:
pip install --upgrade transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 113.3 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.9.0
    Uninstalling accelerate-1.9.0:
      Successfully uninstalled accelerate-1.9.0
Note: you may need to restart the kernel to use updated packages.


In [5]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
import numpy as np
from typing import Dict, List, Any, Tuple, Optional
import time
from datetime import datetime
import re
from tqdm import tqdm
import faiss
from sklearn.preprocessing import normalize


# ============ CẤU HÌNH THAM SỐ ============
# Điều chỉnh các tham số để tối ưu hiệu suất retrieval
CONFIG = {
    # Tham số semantic search
    "top_k_question": 1,      
    "top_k_keywords": 1,       
    "min_score_threshold": 0.35, 
    
    # Tham số context
    "max_context_chunks": 2,  
    "max_context_length": 1200, 
    
    # Tham số chunking
    "max_chunk_size": 500,    
}


class JSONKnowledgeBaseProcessor:
    """Processor để chuyển đổi dữ liệu JSON SGK thành text chunks"""
    
    def __init__(self, json_path: str):
        self.json_path = json_path
        self.data = self.load_json()
        self.text_chunks = []
        self.metadata = []
        
    def load_json(self) -> List[Dict]:
        """Load dữ liệu JSON"""
        with open(self.json_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    
    def process_content(self, content_list: List[str], metadata: Dict) -> List[Dict]:
        """Xử lý danh sách content thành các chunks"""
        chunks = []
        
        # Nối tất cả content thành một đoạn văn
        full_content = " ".join(content_list)
        
        # Chia thành các chunks nhỏ hơn
        max_chunk_size = CONFIG["max_chunk_size"]
        
        if len(full_content) <= max_chunk_size:
            chunks.append({
                "text": full_content,
                "metadata": metadata.copy()
            })
        else:
            # Chia theo câu
            sentences = re.split(r'(?<=[.!?])\s+', full_content)
            current_chunk = ""
            
            for sentence in sentences:
                if len(current_chunk) + len(sentence) <= max_chunk_size:
                    current_chunk += (" " if current_chunk else "") + sentence
                else:
                    if current_chunk:
                        chunks.append({
                            "text": current_chunk.strip(),
                            "metadata": metadata.copy()
                        })
                    current_chunk = sentence
            
            if current_chunk:
                chunks.append({
                    "text": current_chunk.strip(),
                    "metadata": metadata.copy()
                })
        
        return chunks
    
    def extract_all_chunks(self) -> List[Dict]:
        """Trích xuất tất cả text chunks từ dữ liệu JSON"""
        all_chunks = []
        chunk_id = 0
        
        for item in self.data:
            topic_id = item.get("topic_id", "")
            topic_description = item.get("topic_description", "")
            lesson_id = item.get("lesson_id", "")
            lesson_title = item.get("lesson_title", "")
            
            # Tạo context header cho lesson
            lesson_context = f"{topic_id}: {topic_description} - {lesson_id}: {lesson_title}"
            
            for section in item.get("sections", []):
                section_index = section.get("index", "")
                section_title = section.get("title", "")
                
                for subsection in section.get("subsections", []):
                    subsection_label = subsection.get("label", "")
                    subsection_title = subsection.get("title", "")
                    content_list = subsection.get("content", [])
                    
                    if not content_list:
                        continue
                    
                    # Metadata cho chunk
                    metadata = {
                        "topic_id": topic_id,
                        "topic_description": topic_description,
                        "lesson_id": lesson_id,
                        "lesson_title": lesson_title,
                        "section_index": section_index,
                        "section_title": section_title,
                        "subsection_label": subsection_label,
                        "subsection_title": subsection_title,
                        "context": lesson_context
                    }
                    
                    # Xử lý content thành chunks
                    chunks = self.process_content(content_list, metadata)
                    
                    for chunk in chunks:
                        chunk["id"] = chunk_id
                        chunk["type"] = "content"
                        all_chunks.append(chunk)
                        chunk_id += 1
        
        print(f"Extracted {len(all_chunks)} text chunks from JSON")
        return all_chunks


class GraphRAGSystemJSON:
    """GraphRAG System sử dụng dữ liệu JSON thay vì Knowledge Graph"""
    
    def __init__(self, json_path: str, 
                 model_name: str = "Qwen/Qwen2.5-1.6B", 
                 embedding_model: str = "BAAI/bge-small-en-v1.5",
                 use_rerank: bool = False):
        print("Initializing GraphRAG System...")
        
        # Load và xử lý dữ liệu JSON
        print("Loading and processing JSON knowledge base...")
        self.kb_processor = JSONKnowledgeBaseProcessor(json_path)
        self.text_chunks = self.kb_processor.extract_all_chunks()
        
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.gpu_count = torch.cuda.device_count()
        self.use_rerank = use_rerank
        print(f"Using device: {self.device}")
        
        # Load models
        print("Loading language model...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if not self.tokenizer.pad_token:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )
        
        # Load embedding model
        print("Loading embedding model...")
        self.embedding_tokenizer = AutoTokenizer.from_pretrained(embedding_model, trust_remote_code=True)
        self.embedding_model = AutoModel.from_pretrained(
            embedding_model,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
        
        # Build entity index từ text chunks
        print("Building entity index...")
        self.entity_index = self.build_entity_index()
        
        # Compute embeddings
        print("Computing chunk embeddings...")
        self.chunk_embeddings, self.faiss_index = self.compute_chunk_embeddings_faiss()
        
        print("GraphRAG System initialized successfully!")
        print(f"  - Config: top_k_question={CONFIG['top_k_question']}, min_score={CONFIG['min_score_threshold']}")
    
    def build_entity_index(self) -> Dict[str, Any]:
        """Xây dựng index từ các text chunks"""
        entity_index = {}
        
        for chunk in self.text_chunks:
            chunk_id = chunk["id"]
            text = chunk["text"].lower()
            
            # Extract keywords từ text
            words = re.findall(r'\w+', text)
            for word in words:
                if len(word) > 3:  # Chỉ lấy các từ có độ dài > 3
                    if word not in entity_index:
                        entity_index[word] = []
                    entity_index[word].append(chunk_id)
        
        return entity_index
    
    def compute_chunk_embeddings_faiss(self, batch_size: int = 64):
        """Tính embedding cho tất cả chunks và tạo FAISS index"""
        all_embeddings = []
        
        print(f"    Computing embeddings for {len(self.text_chunks)} chunks in batches of {batch_size}...")
        
        for i in tqdm(range(0, len(self.text_chunks), batch_size), desc="Embedding chunks"):
            batch_chunks = self.text_chunks[i:i+batch_size]
            batch_texts = [chunk["text"] for chunk in batch_chunks]
            
            inputs = self.embedding_tokenizer(
                batch_texts, 
                padding=True, 
                truncation=True, 
                max_length=512,
                return_tensors="pt"
            ).to(self.device)
            
            with torch.no_grad():
                outputs = self.embedding_model(**inputs)
                embeddings = outputs.last_hidden_state.mean(dim=1)
                embeddings = embeddings.cpu().numpy()
            
            all_embeddings.append(embeddings)
        
        all_embeddings = np.vstack(all_embeddings)
        all_embeddings = normalize(all_embeddings)
        
        # Create FAISS index
        dimension = all_embeddings.shape[1]
        index = faiss.IndexFlatIP(dimension)
        index.add(all_embeddings.astype('float32'))
        
        return all_embeddings, index
    
    def semantic_search(self, query: str, top_k: int = 5) -> List[Dict]:
        """Tìm kiếm semantic trong text chunks với ngưỡng điểm"""
        inputs = self.embedding_tokenizer(
            query, 
            padding=True, 
            truncation=True, 
            max_length=512,
            return_tensors="pt"
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.embedding_model(**inputs)
            query_embedding = outputs.last_hidden_state.mean(dim=1)
            query_embedding = query_embedding.cpu().numpy()
        
        query_embedding = normalize(query_embedding)
        
        scores, indices = self.faiss_index.search(query_embedding.astype('float32'), top_k)
        
        results = []
        min_score = CONFIG["min_score_threshold"]
        
        for idx, score in zip(indices[0], scores[0]):
            # Áp dụng ngưỡng điểm tối thiểu
            if idx < len(self.text_chunks) and score >= min_score:
                chunk = self.text_chunks[idx].copy()
                chunk["score"] = float(score)
                results.append(chunk)
        
        return results
    
    def analyze_question_fast(self, question: str) -> Dict:
        """Phân tích nhanh câu hỏi"""
        question_lower = question.lower()
        
        # Stopwords tiếng Việt
        stop_words = {'là', 'và', 'của', 'có', 'được', 'trong', 'cho', 'đã', 'với', 'này', 'đó', 'hay', 'không'}
        
        words = [word for word in re.findall(r'\w+', question_lower) if word not in stop_words and len(word) > 1]
        keywords = " ".join(words[:5])
        
        # Simple pattern matching
        subject = ""
        action = ""
        obj = ""
        
        patterns = {
            "ai": "person",
            "cái gì": "thing", 
            "ở đâu": "location",
            "khi nào": "time",
            "tại sao": "reason",
            "như thế nào": "method",
            "bao nhiêu": "quantity"
        }
        
        for pattern, category in patterns.items():
            if pattern in question_lower:
                action = category
                break
        
        return {
            "subject": subject,
            "action": action,
            "object": obj,
            "keywords": keywords,
            "original": question
        }
    
    def retrieve_relevant_context(self, question: str, analysis: Dict) -> str:
        """Truy xuất ngữ cảnh liên quan với các tham số được tối ưu"""
        # Tìm theo câu hỏi với top_k được cấu hình
        chunks = self.semantic_search(question, top_k=CONFIG["top_k_question"])
        
        # Tìm theo keywords với top_k được cấu hình
        if analysis["keywords"] and analysis["keywords"].strip():
            keyword_chunks = self.semantic_search(analysis["keywords"], top_k=CONFIG["top_k_keywords"])
            chunks.extend(keyword_chunks)
        
        # Remove duplicates
        seen_texts = set()
        unique_chunks = []
        for chunk in chunks:
            if chunk["text"] not in seen_texts:
                seen_texts.add(chunk["text"])
                unique_chunks.append(chunk)
        
        # Sắp xếp theo điểm và giới hạn số chunks
        unique_chunks.sort(key=lambda x: x.get("score", 0), reverse=True)
        max_chunks = CONFIG["max_context_chunks"]
        
        # Combine context with metadata
        context_parts = []
        total_length = 0
        max_length = CONFIG["max_context_length"]
        
        for chunk in unique_chunks[:max_chunks]:
            metadata = chunk.get("metadata", {})
            context_header = f"[{metadata.get('lesson_id', '')} - {metadata.get('lesson_title', '')}]"
            if metadata.get('section_title'):
                context_header += f" > {metadata.get('section_title', '')}"
            if metadata.get('subsection_title'):
                context_header += f" > {metadata.get('subsection_title', '')}"
            
            chunk_text = f"{context_header}\n{chunk['text']}"
            
            # Kiểm tra giới hạn độ dài
            if total_length + len(chunk_text) > max_length:
                # Cắt ngắn nếu vượt giới hạn
                remaining = max_length - total_length
                if remaining > 100:  # Còn đủ chỗ cho ít nhất 100 ký tự
                    context_parts.append(chunk_text[:remaining] + "...")
                break
            
            context_parts.append(chunk_text)
            total_length += len(chunk_text)
        
        return "\n\n".join(context_parts) if context_parts else "Không tìm thấy thông tin liên quan."
    
    def answer_question(self, question: str, options: List[Dict]) -> Tuple[str, str, float]:
        """Trả lời câu hỏi multiple choice"""
        analysis = self.analyze_question_fast(question)
        context = self.retrieve_relevant_context(question, analysis)
        
        # Prepare options text
        options_text = ""
        option_letters = ["A", "B", "C", "D"]
        for i, (option, letter) in enumerate(zip(options, option_letters)):
            options_text += f"{letter}. {option['answer']}\n"
        
        # Create prompt
        prompt = f"""Dựa vào kiến thức được đưa. Kiểm tra thông tin sau và trả lời câu hỏi. Nếu thiếu thông tin hoặc không rõ ràng trả lời bằng dữ liệu hiện có.

THÔNG TIN:
{context}

CÂU HỎI: {question}

CÁC ĐÁP ÁN:
{options_text}

Dựa vào thông tin trên, chọn đáp án phù hợp nhất.

ĐÁP ÁN:"""
        
        start_time = time.time()
        
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(self.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        processing_time = time.time() - start_time
        
        # Extract answer
        answer_part = response.split("ĐÁP ÁN:")[-1].strip()
        answer = ""
        for char in answer_part:
            if char.upper() in "ABCD":
                answer = char.upper()
                break
        
        return answer, context, processing_time
    
    def answer_true_false(self, question: str) -> Tuple[str, str, float]:
        """Trả lời câu hỏi True/False"""
        analysis = self.analyze_question_fast(question)
        context = self.retrieve_relevant_context(question, analysis)
        
        prompt = f"""Bạn là một chuyên gia về lịch sử Việt Nam và thế giới. Hãy đọc kỹ phát biểu sau và xác định xem nó ĐÚNG hay SAI.

Phát biểu: {question}

Hãy phân tích ngắn gọn và đưa ra kết luận. Trả lời bằng "Đúng" nếu phát biểu đúng, hoặc "Sai" nếu phát biểu sai.

Kết luận:"""
        
        start_time = time.time()
        
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(self.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        processing_time = time.time() - start_time
        
        # Extract answer
        answer_part = response.split("TRẢ LỜI:")[-1].strip().lower()
        
        if "đúng" in answer_part or "true" in answer_part:
            answer = "Đúng"
        elif "sai" in answer_part or "false" in answer_part:
            answer = "Sai"
        else:
            answer = ""
        
        return answer, context, processing_time


class QuestionEvaluator:
    """Evaluator để đánh giá kết quả trả lời câu hỏi"""
    
    def __init__(self, rag_system: GraphRAGSystemJSON):
        self.rag_system = rag_system
        self.results = {
            "multiple_choice": {"correct": 0, "total": 0, "no_prediction": 0, "details": []},
            "true_false": {"correct": 0, "total": 0, "no_prediction": 0, "details": []},
            "overall": {}
        }
    
    def evaluate_multiple_choice(self, questions: List[Dict]) -> Dict:
        """Đánh giá câu hỏi multiple choice"""
        correct = 0
        total = len(questions)
        no_prediction = 0
        total_time = 0
        
        print(f"Evaluating {total} multiple choice questions...")
        
        for i, q in enumerate(tqdm(questions, desc="Multiple Choice")):
            question = q["question"]
            options = q["options"]
            
            # Find correct answer
            correct_answer = None
            option_letters = ["A", "B", "C", "D"]
            for j, opt in enumerate(options):
                if opt.get("isCorrect", False):
                    correct_answer = option_letters[j]
                    break
            
            # Get prediction
            predicted, context, proc_time = self.rag_system.answer_question(question, options)
            total_time += proc_time
            
            if not predicted:
                no_prediction += 1
            elif predicted == correct_answer:
                correct += 1
            
            # Log progress
            if (i + 1) % 5 == 0:
                print(f"   Completed {i+1}/{total} - Accuracy: {correct/(i+1)*100:.1f}% - Avg time: {total_time/(i+1):.1f}s")
        
        accuracy = (correct / total * 100) if total > 0 else 0
        avg_time = total_time / total if total > 0 else 0
        
        self.results["multiple_choice"].update({
            "correct": correct,
            "total": total,
            "no_prediction": no_prediction,
            "accuracy": accuracy,
            "avg_processing_time": avg_time
        })
        
        return self.results["multiple_choice"]
    
    def evaluate_true_false(self, questions: List[Dict]) -> Dict:
        """Đánh giá câu hỏi True/False"""
        correct = 0
        total = len(questions)
        no_prediction = 0
        total_time = 0
        
        print(f"Evaluating {total} true/false questions...")
        
        for i, q in enumerate(tqdm(questions, desc="True/False")):
            question = q["question"]
            options = q["options"]
            
            # Find correct answer
            correct_answer = None
            for opt in options:
                if opt.get("isCorrect", False):
                    correct_answer = opt["answer"]
                    break
            
            # Get prediction
            predicted, context, proc_time = self.rag_system.answer_true_false(question)
            total_time += proc_time
            
            if not predicted:
                no_prediction += 1
            elif predicted == correct_answer:
                correct += 1
            
            # Log progress
            if (i + 1) % 5 == 0:
                print(f"   Completed {i+1}/{total} - Accuracy: {correct/(i+1)*100:.1f}% - Avg time: {total_time/(i+1):.1f}s")
        
        accuracy = (correct / total * 100) if total > 0 else 0
        avg_time = total_time / total if total > 0 else 0
        
        self.results["true_false"].update({
            "correct": correct,
            "total": total,
            "no_prediction": no_prediction,
            "accuracy": accuracy,
            "avg_processing_time": avg_time
        })
        
        return self.results["true_false"]
    
    def evaluate_questions(self, questions_data: Dict) -> Dict:
        """Đánh giá tất cả câu hỏi"""
        start_time = time.time()
        print("Starting evaluation...")
        
        # Evaluate multiple choice
        mc_questions = questions_data.get("multiple_choice", [])
        if mc_questions:
            self.evaluate_multiple_choice(mc_questions)
        
        # Evaluate true/false
        tf_questions = questions_data.get("true_false", [])
        if tf_questions:
            self.evaluate_true_false(tf_questions)
        
        # Calculate overall results
        elapsed = time.time() - start_time
        self.calculate_overall_results(elapsed)
        
        return self.results
    
    def calculate_overall_results(self, elapsed_time: float):
        """Tính toán kết quả tổng thể"""
        mc = self.results["multiple_choice"]
        tf = self.results["true_false"]
        
        total_questions = mc["total"] + tf["total"]
        total_correct = mc["correct"] + tf["correct"]
        total_no_prediction = mc.get("no_prediction", 0) + tf.get("no_prediction", 0)
        
        overall_accuracy = (total_correct / total_questions * 100) if total_questions > 0 else 0
        prediction_rate = ((total_questions - total_no_prediction) / total_questions * 100) if total_questions > 0 else 0
        
        hours, remainder = divmod(elapsed_time, 3600)
        minutes, seconds = divmod(remainder, 60)
        
        self.results["overall"].update({
            "total_questions": total_questions,
            "total_correct": total_correct,
            "total_no_prediction": total_no_prediction,
            "overall_accuracy": overall_accuracy,
            "prediction_rate": prediction_rate,
            "elapsed_time": f"{int(hours)}h {int(minutes)}m {int(seconds)}s",
            "questions_per_minute": total_questions / (elapsed_time / 60) if elapsed_time > 0 else 0,
            "config": CONFIG  # Lưu cấu hình đã dùng
        })
    
    def print_summary(self, results: Dict):
        """In kết quả tóm tắt"""
        print(f"\n{'='*60}")
        print(f"{'FINAL RESULTS':^60}")
        print(f"{'='*60}")
        
        mc = results["multiple_choice"]
        tf = results["true_false"]
        ov = results["overall"]
        
        print(f"MULTIPLE CHOICE: {mc['correct']}/{mc['total']} = {mc.get('accuracy', 0):.1f}%")
        print(f"  Avg time: {mc.get('avg_processing_time', 0):.1f}s per question")
        print(f"TRUE/FALSE:      {tf['correct']}/{tf['total']} = {tf.get('accuracy', 0):.1f}%")
        print(f"  Avg time: {tf.get('avg_processing_time', 0):.1f}s per question")
        print(f"\nOVERALL:         {ov['total_correct']}/{ov['total_questions']} = {ov['overall_accuracy']:.1f}%")
        print(f"TIME:            {ov['elapsed_time']}")
        print(f"SPEED:           {ov.get('questions_per_minute', 0):.1f} questions/minute")
        print(f"{'='*60}\n")

In [6]:
def main():
    # Initialize the RAG system với file JSON
    print("Starting GraphRAG Evaluation System with JSON Knowledge Base...")
    
    # Đường dẫn đến file JSON dữ liệu SGK
    json_path = "/kaggle/input/SGK/SGK_Lich_Su_12_Ket_Noi_Tri_Thuc.json"
    
    rag_system = GraphRAGSystemJSON(
        json_path,
        model_name="Qwen/Qwen3-4B",
        embedding_model="Qwen/Qwen3-Embedding-0.6B",
        use_rerank=False
    )
    
    # Load questions
    print("Loading questions...")
    questions_path = "/kaggle/input/question_1000.json"
    
    with open(questions_path, 'r', encoding='utf-8') as f:
        questions_data = json.load(f)
    
    evaluator = QuestionEvaluator(rag_system)
    
    print(f"Total questions loaded:")
    print(f"   - Multiple Choice: {len(questions_data.get('multiple_choice', []))}")
    print(f"   - True/False: {len(questions_data.get('true_false', []))}")
    
    results = evaluator.evaluate_questions(questions_data)
    evaluator.print_summary(results)
    
    # Save results
    with open('/kaggle/working/evaluation_results_json.json', 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print("Results saved to /kaggle/working/evaluation_results_json.json")


if __name__ == "__main__":
    main()

Starting GraphRAG Evaluation System with JSON Knowledge Base...
Initializing GraphRAG System...
Loading and processing JSON knowledge base...
Extracted 340 text chunks from JSON
Using device: cuda
Loading language model...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

   Completed 480/488 - Accuracy: 52.5% - Avg time: 1.7s


True/False:  99%|█████████▉| 485/488 [15:07<00:05,  1.88s/it]

   Completed 485/488 - Accuracy: 52.8% - Avg time: 1.7s


True/False: 100%|██████████| 488/488 [15:12<00:00,  1.87s/it]


                       FINAL RESULTS                        
MULTIPLE CHOICE: 322/512 = 62.9%
  Avg time: 2.6s per question
TRUE/FALSE:      258/488 = 52.9%
  Avg time: 1.7s per question

OVERALL:         580/1000 = 58.0%
TIME:            0h 38m 25s
SPEED:           26.0 questions/minute

Results saved to /kaggle/working/evaluation_results_json.json
